In [9]:
import numpy as np
import gymnasium as gym

# Load the environment
env = gym.make("MountainCar-v0")

# Number of bins
num_bins = 20

# Create bins
pos_bins = np.linspace(-1.2, 0.6, num_bins)
vel_bins = np.linspace(-0.07, 0.07, num_bins)

# Initialize Q-tables
Q1 = np.zeros((num_bins, num_bins, 3))
Q2 = np.zeros((num_bins, num_bins, 3))

# Hyperparameters
alpha = 0.1
gamma = 0.99
epsilon = 0.1

# Convert continuous state to discrete state
def get_discretized_state(state):
    pos, vel = state

    pos_idx = np.digitize(pos, pos_bins) - 1
    vel_idx = np.digitize(vel, vel_bins) - 1

    pos_idx = np.clip(pos_idx, 0, num_bins - 1)
    vel_idx = np.clip(vel_idx, 0, num_bins - 1)

    return int(pos_idx), int(vel_idx)

# Train for 1000 episodes
for episode in range(1000):

    raw_state, info = env.reset()
    state = get_discretized_state(raw_state)

    done = False

    while not done:

        # Epsilon-greedy action selection
        if np.random.rand() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(Q1[state[0], state[1]] +
                               Q2[state[0], state[1]])

        # Take action
        next_raw_state, reward, terminated, truncated, _ = env.step(action)

        next_state = get_discretized_state(next_raw_state)

        done = terminated or truncated

        # Double Q-learning update
        if np.random.rand() < 0.5:

            best_next_action = np.argmax(
                Q1[next_state[0], next_state[1]]
            )

            target = reward + gamma * Q2[
                next_state[0],
                next_state[1],
                best_next_action
            ]

            Q1[state[0], state[1], action] += alpha * (
                target - Q1[state[0], state[1], action]
            )

        else:

            best_next_action = np.argmax(
                Q2[next_state[0], next_state[1]]
            )

            target = reward + gamma * Q1[
                next_state[0],
                next_state[1],
                best_next_action
            ]

            Q2[state[0], state[1], action] += alpha * (
                target - Q2[state[0], state[1], action]
            )

        state = next_state

print("Double Q-Tables trained successfully!")

start_idx = get_discretized_state([-0.5, 0.0])

print("Sample Q-values for starting position [-0.5, 0.0]:")

print("Table 1:", np.round(Q1[start_idx[0], start_idx[1]], 2))
print("Table 2:", np.round(Q2[start_idx[0], start_idx[1]], 2))

Double Q-Tables trained successfully!
Sample Q-values for starting position [-0.5, 0.0]:
Table 1: [-43.03 -42.85 -42.25]
Table 2: [-42.54 -43.08 -42.25]
